In [1]:
# Celda 1. Base de datos de productos
import pandas as pd

datos = [
    {'codigo': 'A001', 'nombre': 'Arroz 1 kg', 'categoria': 'Abarrotes', 'precio': 1500, 'stock': 20},
    {'codigo': 'A002', 'nombre': 'Fideos 400 g', 'categoria': 'Abarrotes', 'precio': 900, 'stock': 35},
    {'codigo': 'B001', 'nombre': 'Leche entera 1 L', 'categoria': 'Lácteos', 'precio': 1200, 'stock': 18},
    {'codigo': 'B002', 'nombre': 'Yogur natural', 'categoria': 'Lácteos', 'precio': 850, 'stock': 25},
    {'codigo': 'C001', 'nombre': 'Bebida cola 1.5 L', 'categoria': 'Bebidas', 'precio': 1900, 'stock': 12},
    {'codigo': 'C002', 'nombre': 'Agua mineral 1.5 L', 'categoria': 'Bebidas', 'precio': 1000, 'stock': 30},
    {'codigo': 'D001', 'nombre': 'Jabón líquido', 'categoria': 'Limpieza', 'precio': 2500, 'stock': 10},
    {'codigo': 'D002', 'nombre': 'Esponja', 'categoria': 'Limpieza', 'precio': 700, 'stock': 40}
]

inventario = pd.DataFrame(datos)
carrito = [] #inicializado como lista vacía
display(inventario)

,codigo,nombre,categoria,precio,stock
0,A001,Arroz 1 kg,Abarrotes,1500,20
1,A002,Fideos 400 g,Abarrotes,900,35
2,B001,Leche entera 1 L,Lácteos,1200,18
3,B002,Yogur natural,Lácteos,850,25
4,C001,Bebida cola 1.5 L,Bebidas,1900,12
5,C002,Agua mineral 1.5 L,Bebidas,1000,30
6,D001,Jabón líquido,Limpieza,2500,10
7,D002,Esponja,Limpieza,700,40


In [2]:
#Funciones básicas del carrito

def buscar_productos(inventario, texto, categoria):

  if texto != '':
    nombres = inventario['nombre'].str.lower()
    inventario = inventario[nombres.str.contains(texto.lower())]

  if categoria != 'Todas':
    inventario = inventario[inventario['categoria'] == categoria]

  return inventario

def agregar_al_carrito(inventario, carrito, codigo, cantidad):
  producto_inventario = inventario[inventario['codigo'] == codigo]
  if len(producto_inventario) == 0:
    return ' No se reconoce el producto.'
  producto_inventario = producto_inventario.iloc[0]

  if cantidad <= 0:
    return 'La cantidad debe ser mayor que 0.'

  cantidad_actual = 0
  for producto in carrito:
    if producto['codigo'] == codigo:
      cantidad_actual = producto['cantidad']

  if cantidad_actual + cantidad > producto_inventario['stock']:
    return 'No hay suficiente stock disponible'

  if cantidad_actual > 0:
    for producto in carrito:
      if producto['codigo'] == codigo:
        producto['cantidad'] = producto['cantidad'] + cantidad

  else:
    carrito.append({
        'codigo': producto_inventario['codigo'],
        'nombre': producto_inventario['nombre'],
        'precio': producto_inventario['precio'],
        'cantidad': cantidad
    })

  return 'Producto agregado al carrito'


def quitar_del_carrito(carrito, codigo):
  for producto in carrito:
    if producto['codigo'] == codigo:
      carrito.remove(producto)
      return 'Producto eliminado del carrito'

def calcular_total(carrito):
  total = 0
  for producto in carrito:
    total += producto['precio'] * producto['cantidad']
  return total

def mostrar_carrito(carrito):
  if len(carrito) == 0:
    return 'El carrito esta vacío'
  tabla = pd.DataFrame(carrito)
  tabla['subtotal'] = tabla['precio'] * tabla['cantidad']
  tabla = tabla[['codigo', 'nombre', 'precio', 'cantidad', 'subtotal']]
  return tabla.to_html(index = False) + '<h3> Total: $' + str(calcular_total(carrito)) + '</h3>'

In [3]:
# Componentes de la interfaz
import ipywidgets as widgets
from IPython.display import display

widget_titulo = widgets.HTML(value='<h1>Carrito de compra-Minimarket</h1><p>Selecciona productos y agregalos al carrito</p>')
widget_busqueda = widgets.Text(description = 'Producto', placeholder = 'Ej. Arroz')
widget_categoria = widgets.Dropdown(description = 'Categoría:',
                                    options =['Todas','Abarrotes',
                                              'Bebidas','Lácteos',
                                              'Limpieza'], value = 'Todas')
widget_cantidad = widgets.BoundedIntText(description = 'Cantidad:',
                                         value = 1, min = 1)
widget_codigo_quitar = widgets.Text(description = 'Código:',
                                    placeholder = 'Ej. A001')



In [4]:
#Botones
widget_boton_buscar = widgets.Button(description = 'Buscar',
                                     button_style ='info' )
widget_boton_agregar = widgets.Button(description = 'Agregar al carrito',
                                      button_style = 'success')
widget_boton_mostrar = widgets.Button(description = 'Ver carrito',
                                      button_style = 'info')
widget_boton_quitar = widgets.Button(description = 'Quitar producto')
widget_boton_vaciar = widgets.Button(description = 'Vaciar carrito',
                                     button_style = 'danger')
widget_boton_limpiar = widgets.Button(description = 'Limpiar')

widget_salida = widgets.HTML(value = '')
widget_mensaje = widgets.HTML(value = '')

In [5]:
#Celda 4. Conexión de botones con las funciones

def accion_limpiar(boton):
  widget_busqueda.value = ''
  widget_categoria.value = 'Todas'
  widget_cantidad.value = 1
  widget_codigo_quitar.value = ''
  widget_mensaje.value = ''
  widget_salida.value = ''

def accion_buscar(boton):
  resultados = buscar_productos(inventario, widget_busqueda.value,
                                widget_categoria.value)
  if resultados.empty:
    widget_salida.value = 'No se encontraron productos.'
  else:
    widget_salida.value = resultados.to_html(index = False)

def accion_agregar(boton):
  resultados = buscar_productos(inventario, widget_busqueda.value,
                                widget_categoria.value)
  if len(resultados) == 0:
    widget_mensaje.value = 'Busca un producto antes de agregarlo.'
    return
  codigo = resultados.iloc[0]['codigo']
  widget_mensaje.value = agregar_al_carrito(inventario, carrito, codigo,
                                              widget_cantidad.value)
  widget_salida.value = mostrar_carrito(carrito)

def accion_mostrar(boton):
  widget_salida.value = mostrar_carrito(carrito)

def accion_quitar(boton):
  widget_mensaje.value = quitar_del_carrito(carrito,
                                            widget_codigo_quitar.value)
  widget_salida.value = mostrar_carrito(carrito)

def accion_vaciar(boton):
  carrito.clear()
  widget_mensaje.value ='El carrito fue vaciado'
  widget_salida.value = mostrar_carrito(carrito)

widget_boton_buscar.on_click(accion_buscar)
widget_boton_agregar.on_click(accion_agregar)
widget_boton_mostrar.on_click(accion_mostrar)
widget_boton_quitar.on_click(accion_quitar)
widget_boton_vaciar.on_click(accion_vaciar)
widget_boton_limpiar.on_click(accion_limpiar)





In [6]:
## Organización y visualización con diseño Aesthetic
import ipywidgets as widgets
from IPython.display import display

# Validación e inicialización segura de los widgets para corregir cualquier NameError
try:
    widget_busqueda
except NameError:
    widget_busqueda = widgets.Text(description='Producto', placeholder='Ej. Arroz')

try:
    widget_categoria
except NameError:
    widget_categoria = widgets.Dropdown(description='Categoría:', options=['Todas','Abarrotes','Bebidas','Lácteos','Limpieza'], value='Todas')

try:
    widget_cantidad
except NameError:
    widget_cantidad = widgets.BoundedIntText(description='Cantidad:', value=1, min=1)

try:
    widget_codigo_quitar
except NameError:
    widget_codigo_quitar = widgets.Text(description='Código:', placeholder='Ej. A001')

try:
    widget_boton_buscar
except NameError:
    widget_boton_buscar = widgets.Button(description='Buscar', button_style='info')

try:
    widget_boton_agregar
except NameError:
    widget_boton_agregar = widgets.Button(description='Agregar al carrito', button_style='success')

try:
    widget_boton_mostrar
except NameError:
    widget_boton_mostrar = widgets.Button(description='Ver carrito', button_style='info')

try:
    widget_boton_quitar
except NameError:
    widget_boton_quitar = widgets.Button(description='Quitar producto')

try:
    widget_boton_vaciar
except NameError:
    widget_boton_vaciar = widgets.Button(description='Vaciar carrito', button_style='danger')

try:
    widget_boton_limpiar
except NameError:
    widget_boton_limpiar = widgets.Button(description='Limpiar')

try:
    widget_salida
except NameError:
    widget_salida = widgets.HTML(value='')

try:
    widget_mensaje
except NameError:
    widget_mensaje = widgets.HTML(value='')

# Inyección de estilos CSS para mejorar la apariencia visual de la app y visibilidad de placeholders
styles = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600&display=swap');

    /* Contenedor principal */
    .main-container {
        font-family: 'Poppins', 'Segoe UI', sans-serif;
        background-color: #f8f9fa;
        padding: 25px;
        border-radius: 15px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.05);
        max-width: 850px;
        margin: auto;
    }

    /* Título principal centrado y con estilo */
    .app-title {
        text-align: center;
        margin-bottom: 5px;
    }

    .app-title h1 {
        color: #2c3e50 !important; /* Forzar color legible oscuro y elegante */
        font-weight: 600;
        margin: 0;
    }

    .app-subtitle {
        text-align: center;
        color: #7f8c8d;
        font-size: 1.1em;
        margin-bottom: 25px;
    }

    /* Encabezados de secciones */
    .section-header {
        color: #34495e;
        border-bottom: 2px solid #3498db;
        padding-bottom: 5px;
        margin-top: 20px;
        margin-bottom: 15px;
        font-weight: 600;
    }

    /* Estilización de Inputs, Textareas y Placeholders de ipywidgets */
    input.widget-text, .widget-text input, .widget-text-input, input[type="text"], input[type="number"] {
        color: #2c3e50 !important;
        background-color: #ffffff !important;
        border: 1px solid #ccd1d9 !important;
        border-radius: 6px !important;
        padding: 6px 12px !important;
    }

    /* Asegurar visibilidad del Placeholder */
    ::placeholder, .widget-text input::placeholder, input::placeholder {
        color: #7f8c8d !important;
        opacity: 1 !important; /* Firefox */
    }

    :-ms-input-placeholder {
        color: #7f8c8d !important;
    }

    ::-ms-input-placeholder {
        color: #7f8c8d !important;
    }

    /* Estilos personalizados para las tablas de salida */
    table {
        width: 100%;
        border-collapse: collapse;
        margin: 15px 0;
        font-size: 0.95em;
        border-radius: 8px 8px 0 0;
        overflow: hidden;
        box-shadow: 0 0 10px rgba(0, 0, 0, 0.05);
    }
    thead tr {
        background-color: #3498db;
        color: #ffffff;
        text-align: left;
        font-weight: bold;
    }
    th, td {
        padding: 12px 15px;
    }
    tbody tr {
        border-bottom: 1px solid #dddddd;
    }
    tbody tr:nth-of-type(even) {
        background-color: #f3f3f3;
    }
    tbody tr:last-of-type {
        border-bottom: 2px solid #3498db;
    }

    /* Mensaje de estado estético */
    .status-msg {
        background-color: #ebf5fb;
        border-left: 4px solid #3498db;
        padding: 10px;
        border-radius: 4px;
        color: #2980b9;
        font-weight: 500;
    }
</style>
"""

# Aplicamos los estilos mediante un widget de HTML
widget_estilos = widgets.HTML(value=styles)

# Redefinimos los títulos con clases CSS específicas
widget_titulo = widgets.HTML(value='<div class="app-title"><h1>🛒 Carrito de Compras Minimarket</h1></div><div class="app-subtitle"><p>Selecciona tus productos favoritos de forma rápida y sencilla</p></div>')

# Organizamos las filas añadiendo clases y espaciado (padding/margins)
fila_busqueda = widgets.HBox([widget_busqueda, widget_categoria, widget_boton_buscar])
fila_busqueda.layout.margin = '10px 0px 15px 0px'

fila_carrito = widgets.HBox([widget_cantidad, widget_boton_agregar, widget_boton_mostrar])
fila_carrito.layout.margin = '10px 0px 15px 0px'

fila_quitar = widgets.HBox([widget_codigo_quitar, widget_boton_quitar, widget_boton_vaciar, widget_boton_limpiar])
fila_quitar.layout.margin = '10px 0px 15px 0px'

# Envolvemos el mensaje en un contenedor con estilo
widget_mensaje_con_estilo = widgets.HTML(value='<div class="status-msg">Listo para operar.</div>')

# Vinculamos la actualización del mensaje para que conserve su estilo estético
def actualizar_mensaje(change):
    if change.new:
        widget_mensaje_con_estilo.value = f'<div class="status-msg">{change.new}</div>'

# Observar cambios en el widget_mensaje original de manera segura
try:
    widget_mensaje.observe(lambda change: actualizar_mensaje(change), names='value')
except Exception:
    pass

widget_formulario = widgets.VBox([
    widgets.HTML('<h3 class="section-header">🔍 Buscar Productos</h3>'),
    fila_busqueda,
    widgets.HTML('<h3 class="section-header">🛍️ Agregar al Carrito</h3>'),
    fila_carrito,
    widgets.HTML('<h3 class="section-header">⚙️ Modificar Carrito</h3>'),
    fila_quitar,
    widget_mensaje_con_estilo
])

widget_app = widgets.VBox([
    widget_estilos,
    widget_titulo,
    widget_formulario,
    widgets.HTML('<h3 class="section-header">📊 Resultados / Carrito</h3>'),
    widget_salida
])

# Aplicamos una clase CSS al contenedor principal para redondear bordes y darle sombra
widget_app.add_class('main-container')

display(widget_app)